# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 2: Refresh / Content Opportunity Scoring** → **Ranking / scoring**

The question is "which pages should be reviewed *first*?" — that is a prioritization problem, not a yes/no gate on every page in isolation. A content strategist gets a finite budget of review hours each cycle; the useful output is an ordered queue (score → sort → take top-N), not a flat list of "declining / not declining" labels.

Classification still sits underneath: a model can learn `P(decline)` from page signals, then that probability becomes the ranking score. Clustering would group similar pages but would not tell anyone what to do first. Pure binary classification without a score would also miss the point — with ~900 pages per client on average, "flag everything declining" is not actionable when over half the inventory trends down.

So the ML task type is **ranking / scoring**, with a classifier probability (or an explicit priority score) as the thing we sort on.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (what we score toward):** `is_declining_label` — 1 when the page's traffic trend is down over the trailing window, else 0.

**Where the label comes from:** it is *observed* from the measured trend, not invented by a strategist's priority rule. The raw CSV has `trend_direction` / `trend_pct` (from last-30 vs prev-30 windows); the starter pipeline then defines `is_declining_label = (trend_direction == "down")` in `scripts/01_prepare_features.py`. So the label is a measured outcome in this 90-day snapshot — with one honest caveat: it is still a **proxy** for "should a human look at this page," not proof that a refresh will recover traffic.

**What we actually ship as output:** a continuous **priority / decline-risk score** per page (model probability or calibrated priority), used to rank the client's inventory. Reason codes (stale + declining position, low engagement on a visible page, etc.) ride along so the human knows *why* the page is high in the queue.

**Leakage rule:** `trend_direction` and `trend_pct` define the label, so they are never features — only signals available without knowing the trend (age, freshness, position, engagement, volume, etc.).

In [1]:
import pandas as pd

_df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Target is not a raw CSV column — the pipeline defines it (see scripts/01_prepare_features.py):
# is_declining_label = (trend_direction == "down"). It is observed from the trend windows,
# not a strategist's hand-written rule.
_df["is_declining_label"] = (_df["trend_direction"].str.lower() == "down").astype(int)

print("Target column sketch (first 8 pages):")
print(
    _df[["content_id", "trend_direction", "trend_pct", "is_declining_label"]]
    .head(8)
    .to_string(index=False)
)
print(f"\nObserved base rate P(is_declining_label=1): {_df['is_declining_label'].mean():.3f}")
print("Note: trend_direction / trend_pct define this label -> never use them as features.")

Target column sketch (first 8 pages):
          content_id trend_direction  trend_pct  is_declining_label
content_304f48230142            down      -41.4                   1
content_a1fb4e703a9e            down      -57.7                   1
content_9aa793d4d895            down      -60.9                   1
content_331d6c4de07b          stable      -13.8                   0
content_d99b7a2d90ca            down      -34.7                   1
content_d4084a4bc775            down      -38.9                   1
content_9a34b442b552            down      -92.3                   1
content_a63219c6e95a          stable        0.6                   0

Observed base rate P(is_declining_label=1): 0.542
Note: trend_direction / trend_pct define this label -> never use them as features.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary success metric: precision@K** (I'll use **precision@50** as the headline number).

**What it means:** of the 50 pages the model puts at the top of a client's / holdout queue, what share actually has `is_declining_label = 1`? That matches the real action — a strategist reviews a short top-of-queue each cycle, not the whole inventory.

**What number means "good":** beat both (1) the **base rate** (~54% declining in this starter slice — random ranking would hit ~0.54 precision@50 by chance) and (2) a transparent **rule baseline**. The repo's starter pipeline already shows that gap on this dataset: baseline rules ~0.24 precision@50 vs random forest ~0.74 (`outputs/model_report.md`). I will treat "clearly above base rate *and* above the rule baseline, on a client-holdout split" as the bar for calling the model useful decision-support.

**Secondary (for ranking quality, not the headline):** ROC-AUC — useful for comparing models, but precision@K is the one that maps to "did the top of the queue earn the review hours?"

In [2]:
# What "good" has to beat on this slice: random ranking ≈ base rate.
base_rate = _df["is_declining_label"].mean()
print(f"Base rate (random ranking precision@K expectation): {base_rate:.3f}")
print("Decision bar: precision@50 > base rate AND > transparent rule baseline (starter report: ~0.24).")
print("Secondary ranking-quality check: ROC-AUC on a client-holdout split.")

Base rate (random ranking precision@K expectation): 0.542
Decision bar: precision@50 > base rate AND > transparent rule baseline (starter report: ~0.24).
Secondary ranking-quality check: ROC-AUC on a client-holdout split.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page (content item)** for one client, with trailing-90-day metrics. Not a client, not a day, not a query — a page in the inventory that a strategist could choose to refresh, expand, protect, prune, or monitor.

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
# Same label definition the starter pipeline uses
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Lane 2 slice: page identity + refresh/opportunity signals + the target sketch
lane2_cols = [
    "content_id",
    "client_id",
    "content_age_days",
    "days_since_last_update",
    "freshness_tier",
    "avg_position",
    "engagement_rate",
    "impressions_90d",
    "clicks_90d",
    "trend_direction",   # used only to DEFINE the label — never a feature
    "is_declining_label",  # target / proxy the score is trained against
]

unit = df[lane2_cols].copy()

print("Unit of analysis: one row = one content page")
print(f"Rows: {len(unit):,}  |  clients: {unit['client_id'].nunique()}  |  unique pages: {unit['content_id'].nunique():,}")
print(f"Grain check (duplicate content_id): {(unit['content_id'].duplicated().sum())}")
print(f"Declining-label base rate: {unit['is_declining_label'].mean():.3f}")
print()
print("Sketch of the target column (is_declining_label) next to the observed trend that defines it:")
print(unit.head(10).to_string(index=False))
print()
print("Label value counts:")
print(unit["is_declining_label"].value_counts().sort_index())

Unit of analysis: one row = one content page
Rows: 30,000  |  clients: 32  |  unique pages: 30,000
Grain check (duplicate content_id): 0
Declining-label base rate: 0.542

Sketch of the target column (is_declining_label) next to the observed trend that defines it:
          content_id         client_id  content_age_days  days_since_last_update freshness_tier  avg_position  engagement_rate  impressions_90d  clicks_90d trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc               187                      20           0-30          10.6             5.88             3803          29            down                   1
content_a1fb4e703a9e client_4e07408562               445                      25           0-30          20.3             0.00            15320           7            down                   1
content_9aa793d4d895 client_7f2253d7e2               141                      20           0-30          36.5             0.00            12581          11     

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule *can* start the work — e.g. "if `days_since_last_update` > 180 and `trend_direction` == down, flag it" — but that is too blunt for this decision for three reasons:

1. **Many signals interact.** Decline risk here mixes content age, freshness tier, search position (with `avg_position == 0` meaning "no data," not rank zero), engagement, and volume. A page that is old but still climbing and engaging is not the same priority as a visible page whose CTR and engagement are both soft. Encoding every interaction by hand turns into a brittle forest of if/else branches.
2. **The base rate is high.** Over half the inventory already trends down (~54%). A rule that flags "anything declining" or "anything old" floods the queue; the hard part is *ordering* the review list so limited editor hours hit the pages most worth looking at first.
3. **A plain rule leaves measurable signal on the table.** On this same starter slice, the transparent baseline scores about **0.24 precision@50** while a random forest reaches about **0.74** (and ROC-AUC ~0.63 → ~0.75) — evidence that the pattern is learnable beyond what a short hand-written rule captures (`outputs/model_report.md`).

**The content action the output supports:** the strategist opens the ranked queue for a client, takes the top-N pages (with reason codes), and schedules refresh / expansion / CTR review / engagement review / monitoring for that cycle. ML earns its place as **decision-support ranking**, not as an autopilot that edits pages on its own.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.